## Affiliate / Rebate joins

validating how many affiliates from the affiliate table exist withing this quarter's rebate list using **inner** and **anti** joins on `affiliate_id`.

the anti-join will indicate Silent affiliates (dimension rows with no rebate activity).

In [5]:
from pathlib import Path

import pandas as pd

# Resolve repo root whether the kernel cwd is project root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "bronze").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
print(BRONZE_DIR)
AFFILIATE_CSV = BRONZE_DIR / "synthetic_dim_affiliate.csv"
REBATE_CSV = BRONZE_DIR / "synthetic_fact_rebate.csv"

affiliates = pd.read_csv(AFFILIATE_CSV)
rebates = pd.read_csv(REBATE_CSV)

# Align join key types (rebate IDs may be read as floats if nulls exist)
affiliates["affiliate_id"] = affiliates["affiliate_id"].astype("Int64")
rebates["affiliate_id"] = pd.to_numeric(rebates["affiliate_id"], errors="coerce").astype("Int64")

print(f"Affiliates: {len(affiliates):,} rows")
print(f"Rebates:    {len(rebates):,} rows")

c:\Users\Emora\OneDrive\Documents\Per Scholas One Drive\OneDrive\CCG Rebate Sample Pipeline\data\bronze
Affiliates: 150 rows
Rebates:    5,696 rows


## Inner join

Rebate rows whose `affiliate_id` exists in the affiliate dimension (matched activity).

In [6]:
inner_joined = affiliates.merge(
    rebates,
    on="affiliate_id",
    how="inner",
    suffixes=("_affiliate", "_rebate"),
)

print(f"Inner join: {len(inner_joined):,} row(s)")
print(f"Distinct affiliates with rebate activity: {inner_joined['affiliate_id'].nunique():,}")
inner_joined.head()

Inner join: 5,696 row(s)
Distinct affiliates with rebate activity: 90


,affiliate_id,affiliate_name,parent_id,state,program_tier_id_tier,transaction_id,partner_id,transaction_date,memo,net_amount
0,1001,North Judithbury Auto Body,NaN,GA,5,TXN-100430,101,2025-06-30,Base Program Rebate,121.65
1,1001,North Judithbury Auto Body,NaN,GA,5,TXN-100542,105,2025-06-19,Base Program Rebate,72.90
2,1001,North Judithbury Auto Body,NaN,GA,5,TXN-100553,102,2024-03-29,Base Program Rebate,598.10
3,1001,North Judithbury Auto Body,NaN,GA,5,TXN-100712,103,2024-02-15,Base Program Rebate,413.01
4,1001,North Judithbury Auto Body,NaN,GA,5,TXN-100733,102,2025-11-07,Base Program Rebate,91.66


## Anti-join

Affiliates present in the dimension with **no** matching rebate rows (silent shops).

In [7]:
rebate_affiliate_keys = rebates[["affiliate_id"]].drop_duplicates()

anti_joined = affiliates.merge(
    rebate_affiliate_keys,
    on="affiliate_id",
    how="left",
    indicator=True,
).query('_merge == "left_only"').drop(columns="_merge")

print(f"Anti-join (silent affiliates): {len(anti_joined):,} row(s)")
anti_joined.head()

Anti-join (silent affiliates): 60 row(s)


,affiliate_id,affiliate_name,parent_id,state,program_tier_id_tier
1,1002,Walker Collision Center,NaN,FL,2
6,1007,Hoffman Collision Center,NaN,TN,5
8,1009,Lisatown Auto Body,NaN,NC,5
10,1011,Moore Collision Center,NaN,GA,6
11,1012,Petersonberg Auto Body,NaN,TN,5
